# Baseline: Faster-Whisper-small

Runs baseline inference and Mixed Error Rate evaluation on the original datasets.

See the repository README for setup instructions, datasets, results, and limitations.


In [ ]:
# Install required packages
!pip install -U datasets[audio]
!pip install jiwer accelerate torch torchaudio torchcodec --quiet
!pip install opencc faster-whisper

import gc, torch, jiwer, re, json, os, opencc
import numpy as np
from datasets import load_dataset, Audio, IterableDataset
from tqdm import tqdm
from faster_whisper import WhisperModel

In [ ]:
# =========================
# GPU Setup for Google Colab (GPU ONLY)
# =========================
print(" Checking for GPU availability...")

if not torch.cuda.is_available():
    print("❌ ERROR: No GPU detected!")
    print("Please enable GPU in Google Colab:")
    raise RuntimeError("GPU is required but not available. Please enable GPU in Colab settings.")

device_str = "cuda"
print(f"GPU detected: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")

# Clear any existing GPU cache
torch.cuda.empty_cache()

# =========================
# Load Faster-Whisper Model (GPU Optimized)
# =========================
print("\n Loading Faster-Whisper model...")
model_size = "small"

# Load the model with device and compute type for optimization
# Use float16 for GPU to optimize speed and memory
model = WhisperModel(model_size, device=device_str, compute_type="float16")
print(f"Model loaded on GPU")

# =========================
# Load Datasets (Streaming for Memory Efficiency)
# =========================
print("\n📁 Loading datasets with streaming...")

# Dataset 1: ML2021_ASR_ST
print("Loading ML2021_ASR_ST...")
try:
    ds1 = load_dataset(
        "ky552/ML2021_ASR_ST",
        split="test",
        streaming=True,
    ).cast_column("audio", Audio(sampling_rate=16000))
    print("✅ ML2021_ASR_ST loaded")
except Exception as e:
    print(f"❌ Error loading ML2021_ASR_ST: {e}")
    ds1 = None

# Dataset 2: CAiRE/ASCEND
print("Loading CAiRE/ASCEND...")
try:
    ds2 = load_dataset(
        "CAiRE/ASCEND",
        split="test",
        streaming=True,
    ).cast_column("audio", Audio(sampling_rate=16000))
    print("✅ CAiRE/ASCEND loaded")
except Exception as e:
    print(f"❌ Error loading CAiRE/ASCEND: {e}")
    ds2 = None

# =========================
# MER Calculation Function
# =========================

# Initialize OpenCC converters for Chinese conversion needs
converter_s2t = opencc.OpenCC('s2t')  # Simplified to Traditional
converter_t2s = opencc.OpenCC('t2s')  # Traditional to Simplified

def compute_mer(ref, hyp, dataset_name, debug=False):
    """
    Calculate Mixed Error Rate with simplified dataset-specific Chinese handling
    """
    try:
        # Clean and normalize text
        ref = ref.strip()
        hyp = hyp.strip()

        # Simple dataset-based Chinese handling
        if dataset_name == "ML2021_ASR_ST":
            # Taiwanese dataset: Use Traditional Chinese standard
            ref_conv = ref
            hyp_conv = converter_s2t.convert(hyp)
            standard_label = "Traditional (Taiwan)"
        elif dataset_name == "CAiRE/ASCEND":
            # Chinese dataset: Use Simplified Chinese standard
            ref_conv = ref
            hyp_conv = converter_t2s.convert(hyp)
            standard_label = "Simplified (China)"
        else:
            ref_conv = ref
            hyp_conv = hyp
            standard_label = "Unknown"

        if debug:
            print(f"      Dataset: {dataset_name} ({standard_label})")
            print(f"      Ref after conversion: '{ref_conv}'")
            print(f"      Hyp after conversion: '{hyp_conv}'")

        # Separate Chinese and English
        chinese_ref = re.sub(r"[A-Za-z0-9\s]", "", ref_conv).strip()
        chinese_hyp = re.sub(r"[A-Za-z0-9\s]", "", hyp_conv).strip()
        english_ref = re.sub(r"[^A-Za-z\s]", "", ref_conv).strip()
        english_hyp = re.sub(r"[^A-Za-z\s]", "", hyp_conv).strip()

        # Remove extra space
        english_ref = ' '.join(english_ref.split())
        english_hyp = ' '.join(english_hyp.split())

        if debug:
            chinese_standard = "Traditional" if dataset_name == "ML2021_ASR_ST" else "Simplified"
            print(f"      Chinese Ref ({chinese_standard}): '{chinese_ref}' (len={len(chinese_ref)})")
            print(f"      Chinese Hyp ({chinese_standard}): '{chinese_hyp}' (len={len(chinese_hyp)})")
            print(f"      English Ref: '{english_ref}' (words={len(english_ref.split()) if english_ref else 0})")
            print(f"      English Hyp: '{english_hyp}' (words={len(english_hyp.split()) if english_hyp else 0})")

        # Chinese characters error rate calculation
        if chinese_ref or chinese_hyp:
            if not chinese_ref and chinese_hyp:
                ins_c = len(chinese_hyp)
                del_c = 0
                sub_c = 0
                n_c = len(chinese_hyp)
            elif chinese_ref and not chinese_hyp:
                ins_c = 0
                del_c = len(chinese_ref)
                sub_c = 0
                n_c = len(chinese_ref)
            else:
                cer_measures = jiwer.process_characters([chinese_ref], [chinese_hyp])
                ins_c = cer_measures.insertions
                del_c = cer_measures.deletions
                sub_c = cer_measures.substitutions
                n_c = len(chinese_ref)
        else:
            ins_c, del_c, sub_c, n_c = 0, 0, 0, 0

        # English word error rate calculation
        if english_ref or english_hyp:
            if not english_ref and english_hyp:
                ins_w = len(english_hyp.split())
                del_w = 0
                sub_w = 0
                n_w = len(english_hyp.split())
            elif english_ref and not english_hyp:
                ins_w = 0
                del_w = len(english_ref.split())
                sub_w = 0
                n_w = len(english_ref.split())
            else:
                wer_measures = jiwer.process_words([english_ref], [english_hyp])
                ins_w = wer_measures.insertions
                del_w = wer_measures.deletions
                sub_w = wer_measures.substitutions
                n_w = len(english_ref.split())
        else:
            ins_w, del_w, sub_w, n_w = 0, 0, 0, 0

        if debug:
            print(f"      Chinese errors: I={ins_c}, D={del_c}, S={sub_c}, N={n_c}")
            print(f"      English errors: I={ins_w}, D={del_w}, S={sub_w}, N={n_w}")

        total_ins = ins_c + ins_w
        total_del = del_c + del_w
        total_sub = sub_c + sub_w
        total_ref = n_c + n_w

        if total_ref == 0:
            return 100.0 if (chinese_hyp or english_hyp) else 0.0

        mer = (total_ins + total_del + total_sub) / total_ref * 100

        if debug:
            print(f"      Final: Total_errors={total_ins+total_del+total_sub}, Total_ref={total_ref}, MER={mer:.2f}%")

        return mer

    except Exception as e:
        print(f"    ❌ MER calculation error: {e}")
        return 100.0

# =========================
# Audio Processing with Faster-Whisper
# =========================
def process_audio_faster_whisper(audio_array):
    """Process audio with faster-whisper model"""
    try:
        segments, info = model.transcribe(audio_array, beam_size=5)

        full_text = ""
        for segment in segments:
            full_text += segment.text

        return full_text

    except Exception as e:
        print(f"Error in Faster-Whisper audio processing: {e}")
        return ""

# =========================
# Dataset Evaluation Function
# =========================
def evaluate_dataset(dataset, dataset_name, text_key, debug_samples=5):
    """
    Evaluate dataset with debug output for first few samples
    debug_samples: number of samples to show detailed debug info
    """
    if dataset is None:
        print(f" Skipping {dataset_name} - not loaded")
        return []

    print(f"\n Processing {dataset_name} (full streaming with debug for first {debug_samples} samples)...")
    mers = []
    processed_count = 0

    for i, sample in enumerate(tqdm(dataset, desc=f"Evaluating {dataset_name}", unit="sample")):
        try:
            # Get audio and reference
            audio_array = sample["audio"]["array"]
            ref_text = sample[text_key]

            # Skip if reference text is empty
            if not ref_text or not ref_text.strip():
                if processed_count < 10: # Only show message for a few skipped samples
                    print(f"    Empty reference text, skipping sample {i+1}")
                continue

            # Show sample info for debug samples
            show_debug = processed_count < debug_samples
            if show_debug:
                print(f"\n  Sample {processed_count + 1} ({dataset_name}) - DEBUG MODE")

            # Process audio with the new function
            pred_text = process_audio_faster_whisper(audio_array)

            if pred_text:
                mer = compute_mer(ref_text, pred_text, dataset_name, debug=show_debug)

                if mer >= 0:
                    mers.append(mer)
                    processed_count += 1

                    if show_debug:
                        print(f"    MER: {mer:.2f}%")
                        print(f"    Reference: '{ref_text}'")
                        print(f"    Hypothesis: '{pred_text}'")

                        if len(ref_text) > 80 or len(pred_text) > 80:
                            print(f"    Ref (short): {ref_text[:80]}{'...' if len(ref_text) > 80 else ''}")
                            print(f"    Hyp (short): {pred_text[:80]}{'...' if len(pred_text) > 80 else ''}")

                else:
                    if show_debug:
                        print(f"    Invalid MER value: {mer:.2f}%, skipping")
            else:
                if show_debug:
                    print(f"    Failed to transcribe sample {i+1}")

        except Exception as e:
            print(f"    Error processing sample {i+1}: {e}")
            continue

        # Garbage collection every 10 samples
        if (i + 1) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        # Show progress every 100 samples
        if (i + 1) % 100 == 0:
            current_avg = sum(mers) / len(mers) if mers else 0
            print(f"  📈 Progress: {len(mers)} samples processed, current avg MER: {current_avg:.2f}%")

    print(f"✅ {dataset_name}: {len(mers)} samples processed successfully")
    return mers

# =========================
# Main Evaluation
# =========================
print("\n Starting evaluation on full streaming datasets...")
print("="*60)

all_mers = []
dataset_results = {}

ml_mers = evaluate_dataset(ds1, "ML2021_ASR_ST", "transcription", debug_samples=5)
all_mers.extend(ml_mers)
dataset_results["ML2021_ASR_ST"] = ml_mers

ascend_mers = evaluate_dataset(ds2, "CAiRE/ASCEND", "transcription", debug_samples=5)
all_mers.extend(ascend_mers)
dataset_results["CAiRE/ASCEND"] = ascend_mers


# =========================
# 📊 Results Summary
# =========================
print("\n" + "="*60)
print("🏆 FINAL RESULTS SUMMARY")
print("="*60)

if all_mers:
    # Overall statistics
    overall_avg = sum(all_mers) / len(all_mers)
    overall_min = min(all_mers)
    overall_max = max(all_mers)

    print(f"📈 Overall Statistics:")
    print(f"   Average MER: {overall_avg:.4f}")
    print(f"   Min MER: {overall_min:.4f}")
    print(f"   Max MER: {overall_max:.4f}")
    print(f"   Total samples: {len(all_mers)}")

    # Per-dataset statistics
    print(f"\n📊 Per-Dataset Results:")
    for dataset_name, mers in dataset_results.items():
        if mers:
            avg_mer = sum(mers) / len(mers)
            print(f"   {dataset_name}: {avg_mer:.4f} (n={len(mers)})")
        else:
            print(f"   {dataset_name}: No valid samples")

    # Success rate
    total_attempted = sum(len(v) for v in dataset_results.values())
    success_rate = len(all_mers) / total_attempted * 100 if total_attempted > 0 else 0
    print(f"\n✅ Success Rate: {len(all_mers)}/{total_attempted} ({success_rate:.1f}%)")

else:
    print("❌ No samples were successfully processed")

print("="*60)

# =========================
# Final Cleanup
# =========================
print("\n🧹 Cleaning up GPU memory...")
torch.cuda.empty_cache()
final_gpu_mem = torch.cuda.memory_allocated(0) // 1024**2
print(f"Final GPU memory: {final_gpu_mem} MB")

gc.collect()
print("✅ Cleanup completed!")
print("🎉 Evaluation finished!")